In [ ]:
!pip install huggingface_hub matplotlib -q

from huggingface_hub import InferenceClient
import matplotlib.pyplot as plt
from PIL import Image
import io

HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxx"  #token is hidden for security

# Models
CHAT_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
ART_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"

# Initialize Client
if HF_TOKEN.startswith("hf_"):
    client = InferenceClient(token=HF_TOKEN)
    print("Setup Complete! Ready for experiments.")
else:
    print("Error: Please insert a valid Hugging Face Token.")

In [ ]:
def test_text_generation(prompt, temp):
    """
    Generates text using the Chat Model with a specific temperature setting.
    """
    print(f"\n--- Testing Temperature: {temp} ---")
    try:
        response = ""
        # API call with temperature parameter
        for token in client.chat_completion(
            [{"role": "user", "content": prompt}],
            model=CHAT_MODEL,
            max_tokens=200,
            temperature=temp,
            stream=True
        ):
            if token.choices[0].delta.content:
                response += token.choices[0].delta.content
        print(f"Output: {response}")
        return response
    except Exception as e:
        print(f"Error: {e}")
        return None

# TEST CASE: Creative Writing
test_prompt = "Once upon a time, a robot fell in love with a toaster. The story continues..."

print(f"PROMPT: {test_prompt}")

# 1. Low Temperature (Deterministic, Logical)
output_low = test_text_generation(test_prompt, 0.1)

# 2. High Temperature (Creative, Random)
output_high = test_text_generation(test_prompt, 0.9)

In [ ]:
def generate_image(prompt_text):
    """
    Generates an image using the Art Model.
    """
    try:
        image = client.text_to_image(prompt_text, model=ART_MODEL)
        return image
    except Exception as e:
        print(f"Error: {e}")
        return None

# 1. SIMPLE PROMPT (Baseline)
simple_prompt = "A cat."
print(f"Generating: {simple_prompt}")
img1 = generate_image(simple_prompt)

# 2. OPTIMIZED PROMPT (Engineered)
optimized_prompt = "A majestic fluffy cat wearing a king's crown, sitting on a velvet throne, cinematic lighting, 8k resolution, hyper-realistic, fantasy style."
print(f"Generating: {optimized_prompt}")
img2 = generate_image(optimized_prompt)

# VISUALIZATION
if img1 and img2:
    fig, axs = plt.subplots(1, 2, figsize=(15, 7))

    # Show Simple Result
    axs[0].imshow(img1)
    axs[0].set_title(f"Simple Prompt: '{simple_prompt}'")
    axs[0].axis("off")

    # Show Optimized Result
    axs[1].imshow(img2)
    axs[1].set_title("Optimized Prompt")
    axs[1].axis("off")

    plt.show()
    print("Comparison displayed successfully.")